In [ ]:
from vpython import *
scene = canvas()
scene.width=800 
scene.height =600 
scene.background= color.gray(0.6)
axes = curve(pos=([(3,0,0),( 4,0,0),(3,0,0),(3, 4,0),( 3,4,0),( 3,0,0),( 3,0,3),( 3,0,3)]), color=color.black)
k=9e9 
scale=5e-3
N=30 
dtheta=2*pi/N
obs=None
quedField = None

class EObs(): #I didn't want to use globals, so I made it with a class def instead.
    def __init__(self,pos = vec(0,0,0),EStart=vec(0,0,0)):
        self.pos = pos #Just so I don't have to reference shape.pos every time
        self.shape = sphere(pos=pos, color=color.white, radius = 0.1)
        self.EStart=EStart
        self.E=EStart
        self.Ring = False
        self.Disk = False
        self.rings = []
        self.Ea=arrow(pos=self.pos,axis=vec(0,0,0),color=vec(1,.8,0),opacity=0.5)
        self.Ei=arrow(pos=self.pos,axis=self.E,color=color.red,opacity=0.5)
    def ERing(self, R=5, ringPos=vec(3,0,0),Q=1e-6,singular=True,speed=(10)):
        self.Ring = True
        dQ=Q/N 
        ri=ring(pos=ringPos,axis=vec(1,0,0),radius=R,color=color.yellow,opacity=0.5,thickness=0.05)
        slice=sphere(pos=ri.pos+vector(0,0,R),radius=0.1,color=color.cyan)
        r=self.pos-slice.pos
        ra=arrow(pos=slice.pos,axis=r,color=color.green,shaftwidth=0.1,opacity=0.5)
        dE=vector (0 ,0 ,0)
        for n in range(N):
            rate(speed) 
            slice.pos=ri.pos+vector(0,R*sin(n*dtheta),R*cos(n*dtheta))
            r=self.pos-slice.pos
            ra.pos=slice.pos
            ra.axis=r 
            dE=k*dQ*norm(r)/mag(r)**2 
            self.E+=dE
            self.Ea.axis=self.EStart+self.E*scale
            scene.title = f"E = {mag(self.E):.2f} N/C, {norm(self.E)}"
        slice.visible=False
        self.rings.append(ri)
        if Q>0:
            ri.color=color.red
        else:
            ri.color=color.blue
        ri.opacity=1
        ra.visible=False
        self.Ring = False
    def EDisk(self,Rf=3,ringPos=vec(3,0,0),Q=1e-6):
        dR=0.05
        R=0
        rings = 0
        self.Disk = True
        while R<Rf:
            if not self.Ring:
                R+=dR
                rings += 1 
                dQ=Q*2*(rings*0.05)*(0.05)/(Rf**2)
                self.ERing(R=R,ringPos=ringPos,Q=dQ,speed=60)
        self.Disk = False
    def ECap(self,disk1Pos=vec(3,0,0),disk2Pos=vec(-3,0,0),Q1=1e-6,Q2=(-1e-6),Rf1=2,Rf2=2):
        self.EDisk(Rf=Rf1, ringPos=disk1Pos, Q=Q1)
        while self.Disk:
            rate(1)
        self.EDisk(Rf=Rf2, ringPos=disk2Pos, Q=Q2)

def click(evt):
    global obs
    pos=scene.mouse.pos
    obs=EObs(pos=pos)
    scene.unbind('mousedown',click)

print("Please note that additional variables (like altered charge magnitudes, positions, sizes or initial fields) can be changed by interacting with the code directly.")
scene.title = f"<b>CLICK TO PLACE OBSERVATION POINT</b>"
#scene.bind('mousemove',move)
scene.bind('mousedown',click)
scene.waitfor('mousedown')
scene.title = f"<b>SELECT A FIELD TYPE</b>"

def Ring(evt):
    global quedField, obs
    for ring in obs.rings:
        ring.visible=False
    obs.E*=0
    obs.Ea.axis*=0
    quedField=lambda:obs.ERing() #This fixes the button problem. Stores function as variable so it can be called later and passes it outside of the button's code.
def Disk(evt):
    global quedField, obs
    for ring in obs.rings:
        ring.visible=False
    obs.E*=0
    obs.Ea.axis*=0
    quedField=lambda:obs.EDisk()
def Cap(evt):
    global quedField, obs
    for ring in obs.rings:
        ring.visible=False
    obs.E*=0
    obs.Ea.axis*=0
    quedField=lambda:obs.ECap()

b1 = button(pos=scene.caption_anchor, text='Ring', bind=Ring)
b2 = button(pos=scene.caption_anchor, text='Disk', bind=Disk)
b3 = button(pos=scene.caption_anchor, text='Capacitor', bind=Cap)
    
while True: 
    rate(60)
    if quedField != None:
        quedField()
        quedField = None


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Please note that additional variables (like altered charge magnitudes, positions, sizes or initial fields) can be changed by interacting with the code directly.
